# Transformers, BERT and GPT — build it in NumPy

**NLP · practice topic 01**

Open `explainer.html` first if you have not. Then come back here and build the thing.

### How to use this notebook

- Every `# TODO` is **one idea**. Write it yourself before scrolling on.
- Boilerplate, tests and printing are already done for you.
- If a test fails, the message tells you what to check.
- `guide.md` has the build plan, the five checkpoints, and the traps.

### What you need

Only **NumPy**. No `torch`, no `transformers`, no downloads, no GPU.
That is deliberate — `BertModel.from_pretrained()` is one line that hides everything
you are trying to learn, and it costs ~2 GB of RAM to run.

By the end you will have written the attention mechanism that sits inside every modern
language model, and demonstrated in code the single difference between BERT and GPT.

In [ ]:
import numpy as np

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)

# Our toy sentence. Five tokens is small enough to read every number.
TOKENS = ["The", "cat", "sat", "on", "it"]
n = len(TOKENS)          # sequence length
d_model = 8              # embedding size (real models use 768)

# Random "embeddings" - untrained, so patterns will be noise.
# We are verifying the MECHANISM, not the linguistics.
X = np.random.randn(n, d_model)

print(f"{n} tokens, each a vector of {d_model} numbers")
print("X.shape =", X.shape)

---
## Helper: a text heatmap

No matplotlib installed, so here is a printer that draws attention matrices with block
characters. Read this one — do not write it.

In [ ]:
def heatmap(matrix, labels=None, title=""):
    """Print a matrix as a shaded grid. Row = token looking, column = token looked at."""
    shades = " .:-=+*#%@"
    labels = labels or [str(i) for i in range(matrix.shape[0])]
    w = max(len(l) for l in labels)
    if title:
        print(title)
    print(" " * (w + 2) + "".join(f"{l[:4]:>5}" for l in labels))
    for i, row in enumerate(matrix):
        cells = ""
        for v in row:
            idx = int(np.clip(v, 0, 1) * (len(shades) - 1))
            cells += f"  {shades[idx] * 2} " if v > 0.001 else "   . "
        print(f"{labels[i]:>{w}}  {cells}   " + " ".join(f"{v:.2f}" for v in row))
    print()

# quick demo so you can see what it looks like
heatmap(np.eye(3) * 0.9 + 0.05, ["a", "b", "c"], "demo — an identity-ish matrix")

---
## Step 1 — Softmax, done safely

Softmax turns arbitrary scores into weights that are positive and sum to 1.

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

**The catch:** `np.exp(1000)` overflows to `inf`. Subtracting the maximum first changes
nothing mathematically (the constant cancels top and bottom) but keeps every exponent ≤ 0.

That trick is itself a common interview question.

In [ ]:
def softmax(x, axis=-1):
    """Numerically stable softmax along `axis`."""
    # TODO 1a: subtract the max along `axis`, keeping dimensions so broadcasting works.
    #          Hint: np.max(x, axis=axis, keepdims=True)
    x_shifted = None

    # TODO 1b: exponentiate, then divide by the sum along `axis` (keepdims again!)
    e = None
    return None


# --- checks ---
a = softmax(np.array([1.0, 2.0, 3.0]))
assert a is not None, "softmax returned None - fill in the TODOs"
assert np.isclose(a.sum(), 1.0), f"weights must sum to 1, got {a.sum()}"
assert np.allclose(a, [0.09003057, 0.24472847, 0.66524096]), "values look wrong"

big = softmax(np.array([1000.0, 1001.0, 1002.0]))
assert not np.isnan(big).any(), "overflow! did you subtract the max first?"
assert np.allclose(big, a), "large inputs should give the same answer as small ones"

print("softmax([1, 2, 3])          =", a)
print("softmax([1000, 1001, 1002]) =", big, "  <- no overflow")
print("\nStep 1 passed.")

---
## Step 2 — Scaled dot-product attention

The whole mechanism, in one formula:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Read it left to right:

1. `Q @ K.T` — every query dotted with every key. Result is `(n, n)`: how relevant is each token to each other token.
2. `/ sqrt(d_k)` — stop the scores growing with dimension, or softmax saturates and gradients die.
3. `softmax` — turn scores into weights that sum to 1.
4. `@ V` — each token becomes a weighted blend of the value vectors.

**Mask goes in at step 2.5** — *before* the softmax, by adding a large negative number.
Masking after the softmax is the single most common bug here, because the rows then no
longer sum to 1.

In [ ]:
def attention(Q, K, V, mask=None):
    """
    Q: (n, d_k)   queries
    K: (n, d_k)   keys
    V: (n, d_v)   values
    mask: (n, n) of 1 = allowed, 0 = forbidden, or None

    returns: output (n, d_v), weights (n, n)
    """
    d_k = Q.shape[-1]

    # TODO 2a: raw scores - every query against every key.  shape (n, n)
    scores = None

    # TODO 2b: scale by sqrt(d_k)
    scores = None

    # TODO 2c: if a mask is given, set forbidden positions to a large negative number
    #          BEFORE the softmax. Hint: np.where(mask == 0, -1e9, scores)
    if mask is not None:
        scores = None

    # TODO 2d: softmax over the LAST axis, then multiply by V
    weights = None
    output = None
    return output, weights


# --- checks ---
Q = K = V = X                      # self-attention: all three come from the same input
out, w = attention(Q, K, V)
assert out is not None, "attention returned None - fill in the TODOs"
assert w.shape == (n, n), f"weights should be ({n}, {n}), got {w.shape}"
assert out.shape == (n, d_model), f"output should be ({n}, {d_model}), got {out.shape}"
assert np.allclose(w.sum(axis=-1), 1.0), "each row of weights must sum to 1"

print("weights.shape =", w.shape, "  output.shape =", out.shape)
print("row sums      =", w.sum(axis=-1))
print("\nStep 2 passed.")

In [ ]:
# Look at what you just built.
heatmap(w, TOKENS, "Bidirectional attention (BERT-style) - every token sees every token")

print("Note: these weights are noise, because our embeddings are random.")
print("You are checking the mechanism works, not that it understands English.")

---
## Step 3 — The causal mask. This is GPT.

Everything so far is **BERT**: no mask, every token sees the whole sentence.

To get **GPT**, forbid each token from seeing anything to its right. That is a lower-triangular
matrix of ones — and it is the entire architectural difference between the two families.

```
        The  cat  sat  on   it
 The     1    0    0    0    0      <- sees only itself
 cat     1    1    0    0    0
 sat     1    1    1    0    0
 on      1    1    1    1    0
 it      1    1    1    1    1      <- sees everything before it
```

In [ ]:
def causal_mask(size):
    """Lower-triangular matrix of ones: position i may attend to positions <= i."""
    # TODO 3: one function call. Hint: np.tril
    return None


# --- checks ---
m = causal_mask(n)
assert m is not None, "causal_mask returned None"
assert m.shape == (n, n), f"expected ({n}, {n}), got {m.shape}"
assert m[0].sum() == 1, "the first token should only see itself"
assert m[-1].sum() == n, "the last token should see everything"
assert np.triu(m, k=1).sum() == 0, "nothing above the diagonal may be allowed"

print(m.astype(int))
print("\nStep 3 passed.")

In [ ]:
# The comparison the whole lesson builds to.
out_bert, w_bert = attention(Q, K, V)
out_gpt,  w_gpt  = attention(Q, K, V, mask=causal_mask(n))

heatmap(w_bert, TOKENS, "BERT  - bidirectional: no mask")
heatmap(w_gpt,  TOKENS, "GPT   - causal: the future is exactly zero")

print("Upper triangle of GPT weights sums to:", np.triu(w_gpt, k=1).sum(), " <- exactly 0")
print("First row of GPT weights:", w_gpt[0], " <- token 1 can only attend to itself, so weight = 1.0")
print()
print("Same code. Same maths. One extra argument.")
print("That argument is the difference between a model that understands and a model that generates.")

### Why this matters

- **GPT can generate** because it was never allowed to see the future during training.
  At inference the future genuinely does not exist yet, so training must match.
- **BERT cannot generate**, because if a token can see the answer it is meant to predict,
  next-token prediction is trivially cheating. That is why BERT is trained with *masked
  language modelling* instead — hide 15% of tokens and reconstruct them from both sides.
- **BERT understands better** because its representation of token 3 uses tokens 4 and 5 too.

---
## Step 4 — Positional encoding

Self-attention is **permutation-invariant**: shuffle the input tokens and it computes exactly
the same thing, just reordered. To the raw mechanism, *"dog bites man"* and *"man bites dog"*
are identical.

So position is injected deliberately, using sine and cosine waves at different frequencies:

$$PE_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right) \qquad
  PE_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

Even dimensions get `sin`, odd get `cos`. Low dimensions oscillate fast, high dimensions slowly —
together they act like the digits of a binary clock, giving every position a unique fingerprint.

In [ ]:
def positional_encoding(seq_len, d):
    """Return a (seq_len, d) matrix of sinusoidal positional encodings."""
    pe = np.zeros((seq_len, d))
    position = np.arange(seq_len)[:, None]            # (seq_len, 1)

    # the denominator 10000^(2i/d), one value per PAIR of dimensions
    div_term = np.power(10000.0, np.arange(0, d, 2) / d)   # (d/2,)

    # TODO 4a: even columns (0, 2, 4, ...) get sin(position / div_term)
    pe[:, 0::2] = None

    # TODO 4b: odd columns (1, 3, 5, ...) get cos(position / div_term)
    pe[:, 1::2] = None

    return pe


# --- checks ---
pe = positional_encoding(n, d_model)
assert pe is not None and not np.isnan(pe).any(), "fill in the TODOs"
assert pe.shape == (n, d_model)
assert np.isclose(pe[0, 0], 0.0), "sin(0) should be 0"
assert np.isclose(pe[0, 1], 1.0), "cos(0) should be 1"
assert not np.allclose(pe[0], pe[1]), "different positions must get different vectors"

print(pe.round(3))
print("\nStep 4 passed. Every row is a different fingerprint.")

In [ ]:
# Prove it. We track ONE token - "cat", index 1 - and move it to a different slot.
#
#   order A:  The  cat  sat  on  it      <- "cat" sits in slot 1
#   order B:  cat  The  on   sat it      <- "cat" sits in slot 0
#
# Question: does "cat" come out of attention as the same vector in both cases?

order_a = [0, 1, 2, 3, 4]
order_b = [1, 0, 3, 2, 4]


def representation_of_cat(order, use_pe):
    X_in = X[order]                       # shuffle the TOKENS
    if use_pe:
        X_in = X_in + pe                  # positions are added by SLOT, after shuffling
    out, _ = attention(X_in, X_in, X_in)
    return out[order.index(1)]            # pull out whichever row "cat" landed in


for use_pe in (False, True):
    a = representation_of_cat(order_a, use_pe)
    b = representation_of_cat(order_b, use_pe)
    label = "WITH positional encoding   " if use_pe else "WITHOUT positional encoding"
    print(f"{label}:  identical = {str(np.allclose(a, b)):5}   max difference = {np.abs(a - b).max():.4f}")

print()
print('WITHOUT: identical. Moving "cat" changed nothing - order was completely invisible.')
print('WITH:    different. The model can now tell where the word sits in the sentence.')
print()
print("That is why positional encoding is not optional.")

# From here on we use the position-aware input.
Xp = X + pe

---
## Step 5 — Multi-head attention

One attention pattern is one opinion about what relates to what. Language has many kinds of
relationship at once — subject–verb, adjective–noun, pronoun–referent.

So split the `d_model` dimensions into `h` **heads**, run attention separately in each, then
concatenate and mix.

**Heads are free.** With `d_model = 8` and `h = 2`, each head works in 4 dimensions.
2 × 4 = 8. You get two perspectives for the price of one.

In [ ]:
def multi_head_attention(X, W_q, W_k, W_v, W_o, h, mask=None):
    """
    X: (n, d_model)
    W_q, W_k, W_v, W_o: (d_model, d_model) learned projections
    h: number of heads
    """
    n_, d_model_ = X.shape
    d_head = d_model_ // h
    assert d_model_ % h == 0, "d_model must divide evenly into h heads"

    # project once, then split into heads
    Q_all, K_all, V_all = X @ W_q, X @ W_k, X @ W_v

    heads = []
    for i in range(h):
        s = slice(i * d_head, (i + 1) * d_head)

        # TODO 5a: take this head's slice of Q, K and V (columns s), run `attention`,
        #          and keep only the output (not the weights).
        head_out, _ = None, None
        heads.append(head_out)

    # TODO 5b: concatenate the heads back to (n, d_model), then apply the output projection W_o
    concat = None
    return None


# --- checks ---
h = 2
W_q, W_k, W_v, W_o = (np.random.randn(d_model, d_model) * 0.3 for _ in range(4))

mha = multi_head_attention(Xp, W_q, W_k, W_v, W_o, h)
assert mha is not None, "fill in the TODOs"
assert mha.shape == (n, d_model), f"output should be ({n}, {d_model}), got {mha.shape}"

print("multi-head output shape:", mha.shape, " <- same as single-head; heads split, they do not add")
print("\nStep 5 passed.")

---
## Step 6 — LayerNorm and the full block

A transformer block is:

```
x = LayerNorm(x + SelfAttention(x))     <- sub-layer 1
x = LayerNorm(x + FeedForward(x))       <- sub-layer 2
```

Two things to notice:

- **The residual `+ x`.** Differentiating `F(x) + x` gives `F'(x) + 1`. That `+1` is an
  unobstructed path for the gradient — without it a 96-layer model would not train at all.
- **LayerNorm, not BatchNorm.** LayerNorm normalises across the *features of one token*.
  BatchNorm would normalise across the batch, which breaks when sequences have different
  lengths and when you generate one token at a time.

In [ ]:
def layer_norm(x, eps=1e-6):
    """Normalise across the FEATURE axis (the last one), per sample."""
    # TODO 6a: mean and standard deviation over axis=-1, keepdims=True
    mu = None
    sigma = None

    # TODO 6b: (x - mu) / (sigma + eps)
    return None


def feed_forward(x, W1, b1, W2, b2):
    """Two linear layers with GELU between. Applied to each position independently."""
    gelu = lambda z: 0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z ** 3)))
    return gelu(x @ W1 + b1) @ W2 + b2


# --- checks ---
ln = layer_norm(X)
assert ln is not None, "fill in the TODOs"
assert np.allclose(ln.mean(axis=-1), 0, atol=1e-5), "each row should have mean ~0"
assert np.allclose(ln.std(axis=-1), 1, atol=1e-2), "each row should have std ~1"

print("row means:", ln.mean(axis=-1).round(6))
print("row stds :", ln.std(axis=-1).round(4))
print("\nStep 6a passed.")

In [ ]:
# Parameters for the feed-forward network. Real models expand to 4x the width.
d_ff = 4 * d_model
W1, b1 = np.random.randn(d_model, d_ff) * 0.1, np.zeros(d_ff)
W2, b2 = np.random.randn(d_ff, d_model) * 0.1, np.zeros(d_model)


def transformer_block(x, mask=None):
    """One complete block: attention -> add & norm -> feed-forward -> add & norm."""
    # TODO 7a: multi-head self-attention, then ADD x back, then layer_norm
    attn_out = multi_head_attention(x, W_q, W_k, W_v, W_o, h, mask=mask)
    x = None

    # TODO 7b: feed-forward, then ADD x back, then layer_norm
    ff_out = feed_forward(x, W1, b1, W2, b2)
    x = None

    return x


# --- checks ---
block_out = transformer_block(Xp)
assert block_out is not None, "fill in the TODOs"
assert block_out.shape == Xp.shape, "a block must preserve shape so blocks can stack"
assert np.allclose(block_out.mean(axis=-1), 0, atol=1e-5), "output should be layer-normalised"

print("in :", Xp.shape, " out:", block_out.shape, " <- identical, which is why you can stack them")
print("\nStep 7 passed. You have built a transformer block.")

In [ ]:
# Stack it. BERT-base is 12 of these; GPT-3 is 96.
def encoder(x, n_blocks=6, mask=None):
    for _ in range(n_blocks):
        x = transformer_block(x, mask=mask)
    return x


bert_style = encoder(Xp, n_blocks=6)                       # no mask
gpt_style  = encoder(Xp, n_blocks=6, mask=causal_mask(n))  # causal mask

print("BERT-style output shape:", bert_style.shape)
print("GPT-style  output shape:", gpt_style.shape)
print()
print("How different are the two representations of each token?")
diffs = []
for i, t in enumerate(TOKENS):
    diff = np.abs(bert_style[i] - gpt_style[i]).mean()
    diffs.append(diff)
    print(f"  {t:>5}  mean abs difference = {diff:.3f}")

print()
print(f"Biggest difference:  {TOKENS[int(np.argmax(diffs))]!r}  (the FIRST token)")
print(f"Smallest difference: {TOKENS[int(np.argmin(diffs))]!r}  (the LAST token)")
print()
print("Think about why, it is the whole lesson in one number:")
print("  - The FIRST token loses the most. Bidirectionally it sees all 5 tokens;")
print("    causally it sees only itself. Almost all of its context was taken away.")
print("  - The LAST token loses the least. Causally it can already see everything")
print("    before it, which is nearly the entire sentence anyway.")
print()
print("This is exactly why GPT-style models are weaker at understanding tasks:")
print("early tokens get impoverished representations. And it is why BERT is the")
print("right tool when the whole text is available up front.")

---
## You have now built

| Piece | What it does |
|---|---|
| `softmax` | Scores → weights, without overflowing |
| `attention` | The core: `softmax(QKᵀ/√d_k)·V` |
| `causal_mask` | The one line that turns BERT into GPT |
| `positional_encoding` | Puts word order back into a permutation-invariant mechanism |
| `multi_head_attention` | Several relationship types in parallel, for free |
| `layer_norm` | Stable activations, per token |
| `transformer_block` | attention → add & norm → FFN → add & norm |

Roughly forty lines of real code. Every large language model in existence is this, scaled up
and trained on a great deal of text.

---
## Stretch exercises

1. **Greedy decoding.** Add a final linear layer to vocabulary size, take the `argmax`, append it
   to the sequence, and run again. That loop *is* text generation.
2. **Cross-attention.** Make `Q` come from one sequence and `K`, `V` from another. That is the
   encoder–decoder link used by T5 and the original translation transformer.
3. **Attention entropy.** Compute `-(w * log(w)).sum(axis=-1)` per row. Low entropy means the
   token focused sharply on one place; high entropy means it spread its attention evenly.
4. **Padding mask.** Real batches pad short sentences. Build a mask that hides padding positions,
   and combine it with the causal mask using logical AND.
5. **Count the parameters.** For `d_model=768`, `h=12`, `d_ff=3072`, 12 blocks: how many weights?
   Compare your answer with BERT-base's 110M. What accounts for the rest?

---
## Then, and only then

Install the library and load a real model:

```bash
pip install torch --index-url https://download.pytorch.org/whl/cpu   # ~200 MB, not 2.5 GB
pip install transformers
```

```python
from transformers import AutoModel, AutoTokenizer
model = AutoModel.from_pretrained("distilbert-base-uncased")   # ~265 MB
print(model.config.num_attention_heads, model.config.num_hidden_layers)
```

Every field in that config will now be something you have built by hand.